# PREPROCESSING DATA

In [315]:
# %load_ext autoreload
%autoreload 2

import sys
sys.path.append('../src')

In [316]:
import numpy as np
import pandas as pd
from preprocessing import impute, get_impute_stats, clip, feature_engieneering, remove_first_column, save_processed

In [317]:
train_df = pd.read_csv('../data/raw/cs-training.csv', delimiter=',')
test_df = pd.read_csv('../data/raw/cs-test.csv', delimiter=',')


## Dropping age = 0

In [318]:
train_df = train_df[train_df['age'] >= 18]

assert len(train_df[train_df['age'] < 18]) == 0, "Dropping age < 18 failed!"

## Using median for null values

MonthlyIncome & NumberOfDependents null values replaced with median from age bins 0-30, 30-45, 45-60, 60-75, 75-120

~~Clipping RevolvingUtilizationOfUnsecuredLines columns into (0, 1) - it is impossible to be out of this range, if it would have chance of having over it then i would leave it but it is mathematically impossible~~

In [319]:
train_df = impute(train_df, "MonthlyIncome")
train_df = impute(train_df, "NumberOfDependents")
# train_df = clip(train_df, bounds=(0,1))

assert train_df['MonthlyIncome'].isnull().sum() == 0, "Train MonthlyIncome Imputation failed!"
assert train_df['NumberOfDependents'].isnull().sum() == 0, "Train NumberOfDependents Imputation failed!"
# assert train_df['RevolvingUtilizationOfUnsecuredLines'].between(0, 1).all(), "Train RevolvingUtilizationOfUnsecuredLines Clipping failed!"

### making sure we dont get data leakage - using median from train_df:

In [320]:
test_df = impute(test_df, "MonthlyIncome", get_impute_stats(train_df, "MonthlyIncome"))
test_df = impute(test_df, "NumberOfDependents", get_impute_stats(train_df, "NumberOfDependents"))
# test_df = clip(test_df, bounds=(0,1))

assert test_df['MonthlyIncome'].isnull().sum() == 0, "Test MonthlyIncome Imputation failed!"
assert test_df['NumberOfDependents'].isnull().sum() == 0, "Test NumberOfDependents Imputation failed!"
# assert test_df['RevolvingUtilizationOfUnsecuredLines'].between(0, 1).all(), "Test RevolvingUtilizationOfUnsecuredLines Clipping failed!"

## Lil feature engieneering

MonthlyDebt = DebtRatio * MonthlyIncome

TotalLatePayments = NumberOfTime30-59DaysPastDueNotWorse + NumberOfTime60-89DaysPastDueNotWorse + NumberOfTimes90DaysLate

In [321]:
train_columns_count = len(train_df.columns.to_list())
test_columns_count = len(test_df.columns.to_list())

train_df = feature_engieneering(train_df)
test_df = feature_engieneering(test_df)

assert len(train_df.columns.to_list()) != train_columns_count, "remove_first_column failed on train_df!"
assert len(test_df.columns.to_list()) != train_columns_count, "remove_first_column failed on test_df!"

## Removing first column - indexes

In [322]:
train_df = remove_first_column(train_df)
test_df = remove_first_column(test_df)

assert train_df.iloc(0) != "Unnamed: 0" ,"Removing index column in train failed!"
assert test_df.iloc(0) != "Unnamed: 0" ,"Removing index column in test failed!"

# Saving processed data into csv

In [323]:
save_processed(train_df, '../data/processed/train.csv')
save_processed(test_df, '../data/processed/test.csv')

Saved 149999 rows to ../data/processed/train.csv
Saved 101503 rows to ../data/processed/test.csv


## Summary

| Step | Action | Rationale |
|------|--------|-----------|
| Age filter | Dropped age < 18 | 1 rows with age=0 - likely data entry errors |
| Null imputation | Median per age group | More accurate than global median, avoids data leakage on test set |
| ~~Outlier clipping~~ | ~~RevolvingUtilization - [0,1]~~ | ~~Values outside range are logically impossible~~ |
| Feature engineering | TotalLatePayments, MonthlyDebt, HasDependents | Domain-driven features likely predictive of default risk |